# Week 7 EXERCISE: Suggested price baseline


**Use case for publishers** — predict **suggested retail price (SRP)** for game products from short descriptions (editions, DLC, bundles, subscriptions).

**Flow:**
1. **Data** — Small list of (product description, typical SRP in USD): base games, deluxe editions, DLC, season passes, subs, bundles.
2. **OpenRouter** — Zero-shot (cloud).
3. **Ollama** — Zero-shot (local).
4. **Eval** — MAE, RMSE, R² (same metrics as fine-tuning evals).


**Who it helps**
- **Publisher** — This notebook is a **publisher tool**. It suggests typical suggested retail prices (SRPs) for game products (editions, DLC, bundles, subscriptions) so the business can set or sense-check pricing. The publisher uses it to decide or validate prices, not to advise players.

In [ ]:
import os
import re
import random
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np

load_dotenv(override=True)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
FRONTIER_MODEL = "openai/gpt-4o-mini"

# Ollama (local)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2:1b"  # Run: ollama pull llama3.2:1b
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
OLLAMA_LABEL = f"Ollama ({OLLAMA_MODEL})"
print(f"Using {OLLAMA_LABEL}. Ensure Ollama is running (ollama serve) and model pulled (ollama pull {OLLAMA_MODEL}).")

Game product test data (description → suggested price in USD)

In [ ]:
QUESTION = "What is a typical suggested retail price in USD for this game product (to the nearest dollar)?"

# Game-industry products: editions, DLC, season passes, subs, bundles (typical SRPs)
ITEMS = [
    {"summary": "AAA base game, standard edition, digital (PC/console).", "price": 59.99},
    {"summary": "Deluxe edition: base game + season pass + bonus cosmetics.", "price": 89.99},
    {"summary": "Story expansion DLC, ~10–15 hours, new region and quests.", "price": 24.99},
    {"summary": "Cosmetic DLC: character skin pack, 5 outfits.", "price": 4.99},
    {"summary": "Season pass only (all year-one DLC), sold separately.", "price": 29.99},
    {"summary": "Physical collector's edition: game, steelbook, artbook, statue.", "price": 129.0},
    {"summary": "Monthly subscription, full game library access, one platform.", "price": 14.99},
    {"summary": "Indie game, digital only, single-player, ~8 hours.", "price": 19.99},
    {"summary": "Premium in-game currency pack, mid-tier (e.g. 2000 coins).", "price": 9.99},
    {"summary": "Complete edition: base game + 2 story DLCs, digital bundle.", "price": 79.99},
]

random.seed(42)
test_data = ITEMS.copy()
random.shuffle(test_data)
print(f"Test set: {len(test_data)} game product items")

Predictors and price extraction

In [ ]:
def extract_price(raw: str):
    """Parse first number (int or float) from model output; return None if not found."""
    if not raw or not isinstance(raw, str):
        return None
    m = re.search(r"\$?\s*([0-9]+\.[0-9]*|[0-9]+)", raw.strip())
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return None
    return None


def predict_openrouter(description: str) -> float:
    prompt = f"{QUESTION}\n\nProduct: {description}\n\nReply with only the suggested price in dollars (e.g. 60 or 24.99)."
    r = openrouter_client.chat.completions.create(
        model=FRONTIER_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=20,
    )
    text = (r.choices[0].message.content or "").strip()
    out = extract_price(text)
    return out if out is not None else 0.0


def predict_ollama(description: str) -> float:
    if not ollama_client:
        return 0.0
    try:
        prompt = f"{QUESTION}\n\nProduct: {description}\n\nReply with only the suggested price in dollars (e.g. 60 or 24.99)."
        r = ollama_client.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20,
        )
        text = (r.choices[0].message.content or "").strip()
        p = extract_price(text)
        return p if p is not None else 0.0
    except Exception as e:
        if not getattr(predict_ollama, "_err_printed", False):
            predict_ollama._err_printed = True
            print(f"Ollama error: {type(e).__name__}: {e}. Is Ollama running? Try: ollama serve && ollama pull {OLLAMA_MODEL}")
        return 0.0

predict_ollama._err_printed = False


Metrics (MAE, RMSE, R²) and comparison

In [ ]:
def metrics(predictor, data):
    preds = [predictor(item["summary"]) for item in data]
    truths = [item["price"] for item in data]
    preds = np.array(preds, dtype=float)
    truths = np.array(truths, dtype=float)
    mae = np.mean(np.abs(preds - truths))
    rmse = np.sqrt(np.mean((preds - truths) ** 2))
    ss_res = np.sum((truths - preds) ** 2)
    ss_tot = np.sum((truths - np.mean(truths)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    return {"MAE": mae, "RMSE": rmse, "R2": r2}


m_or = metrics(predict_openrouter, test_data)
print(f"OpenRouter ({FRONTIER_MODEL}): MAE={m_or['MAE']:.1f}, RMSE={m_or['RMSE']:.1f}, R²={m_or['R2']:.3f}")

m_ollama = metrics(predict_ollama, test_data)
print(f"{OLLAMA_LABEL}:  MAE={m_ollama['MAE']:.1f}, RMSE={m_ollama['RMSE']:.1f}, R²={m_ollama['R2']:.3f}")

In [ ]:
print("Sample predictions (first 4 game products):")
for item in test_data[:4]:
    p_or = predict_openrouter(item["summary"])
    p_ollama = predict_ollama(item["summary"])
    ollama_str = "—" if (p_ollama is None or p_ollama == 0.0) else f"${p_ollama:.2f}"
    print(f"  True: ${item['price']:.2f} | OpenRouter: ${p_or:.2f} | {OLLAMA_LABEL}: {ollama_str}")
    print(f"    {item['summary'][:60]}...")